In [1]:
import pandas as pd
from pathlib import Path
# clean_adult.py
from __future__ import annotations
import numpy as np
from pathlib import Path

root = Path.cwd().parent

In [2]:
df = pd.read_csv(f'{root}/data/raw/raw_dataset.csv')
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
df = df.drop(columns=['native-country','sex','race','fnlwgt'])

## Clean numericas

In [4]:


def clean_adult_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Limpieza mínima y robusta para Adult."""
    df = df.copy()

    # 1) Normaliza nombres
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(" ", "_").str.replace("-", "_")
    )

    # 2) Limpia strings: trim y NaN
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].str.strip().replace({"?": np.nan})

    # 3) Target binario
    if "income" in df.columns:
        df["income"] = (
            df["income"]
            .replace({">50K": 1, "<=50K": 0, ">50K.": 1, "<=50K.": 0})
            .astype("Int64")   # permite NA si quedaron faltantes
        )

    # 4) Features engineering
    if {"capital_gain", "capital_loss"}.issubset(df.columns):
        df["capital_net"] = df["capital_gain"].fillna(0) - df["capital_loss"].fillna(0)
        df["has_capital_gain"] = (df["capital_gain"].fillna(0) > 0).astype("Int8")
        df["has_capital_loss"] = (df["capital_loss"].fillna(0) > 0).astype("Int8")

    # 5) Elimina columnas poco útiles o redundantes
    drop_cols = []
    # si existen ambas, quita 'education' (texto) y deja 'education_num' (ordinal)
    if "education" in df.columns and "education_num" in df.columns:
        drop_cols.append("education")
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

    # 6) Tipado
    for c in df.columns:
        if c in {"age","education_num","capital_gain","capital_loss","hours_per_week","capital_net"}:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    # 7) Quita duplicados exactos
    df = df.drop_duplicates(ignore_index=True)

    return df

## Clean categorical (One hot)

In [5]:
def clean_adult(df: pd.DataFrame, *, one_hot: bool = True,
                top_k: int | None = None, min_freq: int | None = None) -> pd.DataFrame:
    df = df.copy()

    # 1) Normaliza nombres
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_").str.replace("-", "_"))

    # 2) Strings: trim, lower y '?' -> NaN
    for c in df.select_dtypes(include="object").columns:
        df[c] = (df[c].astype("string").str.strip().str.lower()
                           .replace({"?": pd.NA}))

    # 3) Target binario
    if "income" in df.columns:
        df["income"] = (df["income"]
                        .replace({">50k": 1, "<=50k": 0, ">50k.": 1, "<=50k.": 0})
                        .astype("Int64"))

    # 4) Feature engineering
    if {"capital_gain","capital_loss"}.issubset(df.columns):
        df["capital_net"] = df["capital_gain"].fillna(0) - df["capital_loss"].fillna(0)
        df["has_capital_gain"] = (df["capital_gain"].fillna(0) > 0).astype("Int8")
        df["has_capital_loss"] = (df["capital_loss"].fillna(0) > 0).astype("Int8")

    # 5) Columnas redundantes
    drop_cols = []
    if "education" in df.columns and "education_num" in df.columns:
        drop_cols.append("education")
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)

    # 6) Tipado numérico
    for c in {"age","education_num","capital_gain","capital_loss","hours_per_week","capital_net"} & set(df.columns):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # 7) Categóricas -> imputación + rare → __other__ + one-hot
    cat_cols = df.select_dtypes(include=["object","string"]).columns.tolist()
    if cat_cols:
        # imputación por moda simple ("missing")
        for c in cat_cols:
            df[c] = df[c].fillna("missing")

            # compactar rarezas (elige UNO de los dos criterios: top_k o min_freq)
            if top_k is not None:
                keep = set(df[c].value_counts().head(top_k).index)
                df[c] = df[c].where(df[c].isin(keep), "__other__")
            elif min_freq is not None:
                vc = df[c].value_counts()
                keep = set(vc[vc >= min_freq].index)
                df[c] = df[c].where(df[c].isin(keep), "__other__")

        if one_hot:
            df = pd.get_dummies(df, columns=cat_cols, drop_first=False,
                                dtype="Int8", prefix_sep="=")

    # 8) Duplicados exactos
    df = df.drop_duplicates(ignore_index=True)
    return df

In [6]:
#df_clean = clean_adult(df, one_hot=True)
df_clean_numeric = clean_adult(df)
df_clean = clean_adult(df_clean_numeric, one_hot=True, top_k=20)    # limita a top-20 por columna

/var/folders/12/07v6cgmx5nv6khqcz213r1b80000gn/T/ipykernel_38378/372168329.py:17: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({">50k": 1, "<=50k": 0, ">50k.": 1, "<=50k.": 0})


In [7]:
pd.set_option('display.max_columns', None)   # sin límite de columnas
pd.set_option('display.width', 0)            # autoajuste
df_clean.head(20)

,age,education_num,capital_gain,capital_loss,hours_per_week,income,capital_net,has_capital_gain,has_capital_loss,workclass=federal_gov,workclass=local_gov,workclass=missing,workclass=never_worked,workclass=private,workclass=self_emp_inc,workclass=self_emp_not_inc,workclass=state_gov,workclass=without_pay,marital_status=divorced,marital_status=married_af_spouse,marital_status=married_civ_spouse,marital_status=married_spouse_absent,marital_status=never_married,marital_status=separated,marital_status=widowed,occupation=adm_clerical,occupation=armed_forces,occupation=craft_repair,occupation=exec_managerial,occupation=farming_fishing,occupation=handlers_cleaners,occupation=machine_op_inspct,occupation=missing,occupation=other_service,occupation=priv_house_serv,occupation=prof_specialty,occupation=protective_serv,occupation=sales,occupation=tech_support,occupation=transport_moving,relationship=husband,relationship=not_in_family,relationship=other_relative,relationship=own_child,relationship=unmarried,relationship=wife
0,39,13,2174,0,40,0,2174,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,50,13,0,0,13,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,38,9,0,0,40,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,53,7,0,0,40,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
4,28,13,0,0,40,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1
5,37,14,0,0,40,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
6,49,5,0,0,16,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
7,52,9,0,0,45,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
8,31,14,14084,0,50,1,14084,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
9,42,13,5178,0,40,1,5178,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [13]:
df_clean.to_csv(f'{root}/data/procesed/procesed_dataset.csv')